# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Structured Streaming (Kakfa producer)** </center>
---
**Profesor**: Pablo Camarillo Ramirez
---
**Student**: Nicolas Navarro Valenzuela 746812

# Create SparkSession

In [1]:
from spark_utils import SparkUtils
from pathlib import Path
import shutil
import pyspark.sql.functions as F

kafka_connector = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0"
su = SparkUtils("Example Kafka", 
                "spark://spark-master:7077",
                spark_packages=kafka_connector)
su.spark


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f1f715b2-44a3-402c-b1ca-d4c96b087bf3;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.0 in central
	found org.apache.kafka#kafka-clients;3.9.0 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0

# Create a data stream from a Kafka topic

In [2]:
# Create the remote connection
kafka_df = su.spark.readStream \
            .format("kafka") \
            .option("kafka.bootstrap.servers", "kafka:9093") \
            .option("subscribe", "kafka-spark-example") \
            .load()

kafka_df.printSchema()

# Transform binary data to string
df_input = kafka_df.selectExpr("CAST(value AS STRING)")

# Clean checkpoint
checkpoint_path = "/opt/spark/work-dir/checkpoints/logs_checkpoint"
dir_path = Path(checkpoint_path)
if dir_path.exists() and dir_path.is_dir():
    shutil.rmtree(dir_path)

words = df_input.select(F.explode(F.split(df_input.value, " ")).alias("word"))
word_count = words.groupBy("word").count()

# Send transformed data to the Sink
query_a = (word_count.writeStream
            .trigger(processingTime='2 second')
            .outputMode("complete")
            .format("console")
            .option("checkpointLocation", checkpoint_path)
            .start())

print("   Press Ctrl+C to stop.\n")
su.spark.streams.awaitAnyTermination()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



26/04/14 01:01:46 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


   Press Ctrl+C to stop.



26/04/14 01:01:49 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 2000} milliseconds, but spent 3100 milliseconds


-------------------------------------------
Batch: 0
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
+----+-----+



...

-------------------------------------------
Batch: 9
-------------------------------------------
+----------+-----+
|      word|count|
+----------+-----+
|   expires|    1|
|       500|    1|
|  detected|    1|
|        in|    1|
|     ERROR|    2|
|2026-04-14|    9|
|       90%|    1|
|  01:02:45|    1|
|  01:02:37|    1|
| completed|    1|
|     spike|    1|
|    Memory|    1|
|  01:03:11|    1|
|  01:02:25|    1|
|       404|    1|
|    Server|    1|
|  Response|    1|
|   cleared|    1|
|       Not|    1|
|     Error|    1|
+----------+-----+
only showing top 20 rows


-------------------------------------------
Batch: 10
-------------------------------------------
+----------+-----+
|      word|count|
+----------+-----+
|   expires|    1|
|       500|    1|
|  detected|    1|
|        in|    1|
|     ERROR|    2|
|2026-04-14|   10|
|       90%|    1|
|  01:02:45|    1|
|  01:02:37|    1|
| completed|    1|
|     spike|    1|
|    Memory|    1|
|  01:03:11|    1|
|  01:02:25|    1|
|       404|    1|
|    Server|    1|
|  Response|    1|
|   Service|    1|
|   cleared|    1|
|       Not|    1|
+----------+-----+
only showing top 20 rows


-------------------------------------------
Batch: 11
-------------------------------------------
+----------+-----+
|      word|count|
+----------+-----+
|   expires|    1|
|       500|    1|
|  detected|    1|
|        in|    1|
|     ERROR|    3|
|2026-04-14|   11|
|       90%|    1|
|  01:02:45|    1|
|  01:02:37|    1|
| completed|    1|
|     spike|    1|
|    Memory|    1|
|  01:03:11|    1|
|  01:02:25|    1|
|       404|    1|
|    Server|    1|
|  Response|    1|
|   Service|    1|
|   cleared|    1|
|       Not|    1|
+----------+-----+
only showing top 20 rows


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

## Custom producer

### Create `server-logs` topic

```
    docker exec -it <Kafka container ID> \
     /opt/kafka/bin/kafka-topics.sh \
      --create --zookeeper zookeeper:2181 \
      --replication-factor 1 --partitions 1 \
      --topic server-logs
```

### Run the producer

```
    docker exec -it <Spark-Notebook container ID> /bin/bash
    # cd src/producers/
    # python3 kafka_producer.py --broker kafka:9093 --topic server-logs --records 20
```

### Run the consumer code

In [ ]:
# Create the remote connection
server_logs_df = (su.spark.readStream
            .format("kafka")
            .option("kafka.bootstrap.servers", "kafka:9093")
            .option("subscribe", "server-logs")
            .load())

# Transform binary data to string
logs_df = server_logs_df.selectExpr("CAST(value AS STRING)")

# Clean checkpoint
checkpoint_path = "/opt/spark/work-dir/checkpoints/logs_checkpoint"
dir_path = Path(checkpoint_path)
if dir_path.exists() and dir_path.is_dir():
    shutil.rmtree(dir_path)

# Transform original dataframe
parsed_df = (
    logs_df
    .withColumn("parts",     F.split(F.col("value"), r" \| "))
    .withColumn("timestamp", F.to_timestamp(F.col("parts")[0], "yyyy-MM-dd HH:mm:ss"))
    .withColumn("level",     F.trim(F.col("parts")[1]))
    .withColumn("message",   F.trim(F.col("parts")[2]))
    .withColumn("server",    F.trim(F.col("parts")[3]))
    .drop("parts", "value")
    .filter(F.col("timestamp").isNotNull())
)

# Write stream in the destination
output_path = "/opt/spark/work-dir/data/streaming/output/"
query_events = (
    parsed_df.writeStream
    .outputMode("append")
    .format("parquet")
    .option("truncate", False)
    .option("checkpointLocation", checkpoint_path)
    .option("path", output_path)
    .partitionBy("server", "level")
    .start()
)

print("   Press Ctrl+C to stop.\n")
su.spark.streams.awaitAnyTermination()

   Press Ctrl+C to stop.



26/04/14 01:00:56 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


StreamingQueryException: [STREAM_FAILED] Query [id = 339296b3-cc23-4dbd-97b2-2e03116479ec, runId = a8a2111a-acb4-4f68-a354-a9ee28519bde] terminated with exception: [CONCURRENT_STREAM_LOG_UPDATE] Concurrent update to the log. Multiple streaming jobs detected for 2.
Please make sure only one streaming job runs on a specific checkpoint location at a time. SQLSTATE: 40000 SQLSTATE: XXKST
=== Streaming Query ===
Identifier: [id = 339296b3-cc23-4dbd-97b2-2e03116479ec, runId = a8a2111a-acb4-4f68-a354-a9ee28519bde]
Current Committed Offsets: {KafkaV2[Subscribe[kafka-spark-example]]: {"kafka-spark-example":{"0":6}}}
Current Available Offsets: {KafkaV2[Subscribe[kafka-spark-example]]: {"kafka-spark-example":{"0":7}}}

Current State: ACTIVE
Thread State: RUNNABLE

Logical Plan:
~WriteToMicroBatchDataSource org.apache.spark.sql.execution.streaming.ConsoleTable$@7a1072f7, 339296b3-cc23-4dbd-97b2-2e03116479ec, [checkpointLocation=/opt/spark/work-dir/checkpoints/logs_checkpoint], Complete
+- ~Aggrega

In [3]:
!ls -lah /opt/spark/work-dir/data/streaming/output/

total 0
drwxr-xr-x 1 root root 512 Apr 14 00:54  .
drwxr-xr-x 1 root root 512 Apr 14 00:49  ..
drwxr-xr-x 1 root root 512 Apr 14 00:53 'server=server-node-1'
drwxr-xr-x 1 root root 512 Apr 14 00:54 'server=server-node-2'
drwxr-xr-x 1 root root 512 Apr 14 00:53 'server=server-node-3'
drwxr-xr-x 1 root root 512 Apr 14 00:53 'server=server-node-4'
drwxr-xr-x 1 root root 512 Apr 14 00:54 'server=server-node-5'
drwxr-xr-x 1 root root 512 Apr 14 00:55  _spark_metadata


In [ ]:
su.spark.stop()

26/04/14 01:23:50 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1001 (kafka/172.18.0.4:9093) could not be established. Node may not be available.
26/04/14 01:23:50 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1001 (kafka/172.18.0.4:9093) could not be established. Node may not be available.
26/04/14 01:23:51 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1001 (kafka/172.18.0.4:9093) could not be established. Node may not be available.
26/04/14 01:23:51 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node 1001 (kafka/172.18.0.4:9093) could not be established. Node may not be available.
26/04/14 01:23:51 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-7be8e4f4-0e3f-4add-9f32-a6feb1d58ff0/pyspark-2d219d79-cd7e-41fc-966f-f0194786b050. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-7be8e4f4-0e3f-4add-9f32-a